In [ ]:
"""
Comfort-first lean control simulation with 2D gain scheduling for Kp and Krate.

Scheduling:
 - Kp and Krate are scheduled smoothly based on (v, |kappa|)
 - Ki remains constant
 - Bilinear interpolation on a small grid ensures no jumps

Everything else preserved from your comfort-first setup:
 - curvature ramp in/out
 - PI outer loop -> rate command
 - rate servo feedback torque
 - feedforward = kphi * phi_des
 - comfort metric: a_lat_feel = g*(phi_des - phi)
"""

import math
import numpy as np
import matplotlib.pyplot as plt


# -----------------------------
# VEHICLE / PLANT PARAMETERS
# -----------------------------
Iphi = 600.0          # [kg*m^2]
kphi = 10000.0        # [N*m/rad]
cphi = 2000.0         # [N*m*s/rad]

m = 1500.0            # [kg] (not used directly in comfort-first)
h = 0.55              # [m]


# -----------------------------
# CONTROLLER BASE (used if scheduling falls to single-point)
# -----------------------------
# Note: we will schedule Kp and Krate; these base values remain as fallbacks
Kp_base = 0.8              # [1/s] (not used directly once scheduling active)
Ki = 0.10                  # [1/s^2] (kept constant)
Krate_base = 6000.0        # [N*m*s/rad] (not used directly once scheduling active)

phi_max_deg = 10.0            # [deg] roll angle limit
phi_dot_cmd_max_deg_s = 6.0   # [deg/s] comfort-first roll-rate command limit


# -----------------------------
# TURN SCENARIO
# -----------------------------
v = 10.0              # [m/s] (scheduling will use this)
R = -200.0            # [m] (+left, -right)

turn_start_time = 1.0  # [s]
turn_end_time   = 10.0 # [s] fully back to straight by this time
turn_ramp_time  = 1.0  # [s] ramp in and ramp out duration (curvature)


# -----------------------------
# GAIN SCHEDULING DEFINITIONS (2D table over v and |kappa|)
# -----------------------------
# grids
v_grid = np.array([5.0, 10.0, 20.0])                       # m/s
kappa_grid = np.array([0.0, 1/400.0, 1/200.0, 1/100.0])    # 1/m

# scheduling tables (rows -> v_grid, cols -> kappa_grid)
Kp_table = np.array([
    [0.5, 0.7, 0.9, 1.1],   # v=5
    [0.6, 0.8, 1.0, 1.2],   # v=10
    [0.8, 1.0, 1.2, 1.4],   # v=20
], dtype=float)

Krate_table = np.array([
    [4000, 5000, 6000, 7500],  # v=5
    [4500, 6000, 7000, 8500],  # v=10
    [6000, 7500, 9000, 11000], # v=20
], dtype=float)

# clamps for safety
Kp_min, Kp_max = 0.3, 2.0
Krate_min, Krate_max = 2000.0, 20000.0


# -----------------------------
# SIMULATION
# -----------------------------
dt = 0.001            # [s]
T = 15.0              # [s]

g = 9.81
phi_max = math.radians(phi_max_deg)
phi_dot_cmd_max = math.radians(phi_dot_cmd_max_deg_s)


def clip(x, lo, hi):
    return max(lo, min(hi, x))

def smoothstep01(x):
    """C1-continuous 0->1 ramp for x in [0,1]."""
    x = clip(x, 0.0, 1.0)
    return x * x * (3.0 - 2.0 * x)


# --- helpers for bilinear interpolation ---
def lerp(a, b, w):
    return a + w * (b - a)

def find_interval(x, grid):
    """
    Returns (i0, i1, w) such that x is between grid[i0] and grid[i1]
    and w in [0,1] is interpolation weight toward i1.
    Clamps to edges.
    """
    if x <= grid[0]:
        return 0, 0, 0.0
    if x >= grid[-1]:
        return len(grid) - 1, len(grid) - 1, 0.0

    for i in range(len(grid) - 1):
        if grid[i] <= x <= grid[i + 1]:
            w = (x - grid[i]) / (grid[i + 1] - grid[i])
            return i, i + 1, w

    return len(grid) - 1, len(grid) - 1, 0.0

def bilinear_2d(x, y, x_grid, y_grid, table):
    """
    Bilinear interpolation over (x_grid, y_grid) for value at (x,y).
    table shape: (len(x_grid), len(y_grid))
    """
    x0, x1, wx = find_interval(x, x_grid)
    y0, y1, wy = find_interval(y, y_grid)

    f00 = table[x0, y0]
    f01 = table[x0, y1]
    f10 = table[x1, y0]
    f11 = table[x1, y1]

    f0 = lerp(f00, f01, wy)
    f1 = lerp(f10, f11, wy)
    return lerp(f0, f1, wx)


# Time
N = int(T / dt) + 1
t = np.linspace(0.0, T, N)

# States and signals
phi = np.zeros(N)          # [rad]
phi_dot = np.zeros(N)      # [rad/s]

phi_des = np.zeros(N)      # [rad]
phi_dot_cmd = np.zeros(N)  # [rad/s]

tau_act = np.zeros(N)      # [N*m]
tau_ff = np.zeros(N)       # [N*m]
tau_fb = np.zeros(N)       # [N*m]

ay = np.zeros(N)           # [m/s^2]
kappa_arr = np.zeros(N)    # [1/m]

e_arr = np.zeros(N)        # [rad]
e_int_arr = np.zeros(N)    # [rad*s]

# Comfort metric: passenger felt lateral acceleration in vehicle frame (reduced-model)
a_lat_feel = np.zeros(N)   # [m/s^2]

# Scheduled gains logging
Kp_used = np.zeros(N)
Krate_used = np.zeros(N)

# PI integrator state
e_int = 0.0


for i in range(N - 1):
    ti = t[i]

    # -----------------------------
    # Comfort-first curvature profile: ramp in, hold, ramp out
    # -----------------------------
    if ti < turn_start_time:
        s = 0.0
    elif ti < turn_start_time + turn_ramp_time:
        xi = (ti - turn_start_time) / turn_ramp_time
        s = smoothstep01(xi)
    elif ti < turn_end_time - turn_ramp_time:
        s = 1.0
    elif ti < turn_end_time:
        xi = (ti - (turn_end_time - turn_ramp_time)) / turn_ramp_time
        s = 1.0 - smoothstep01(xi)
    else:
        s = 0.0

    # Curvature ramps from 0 -> 1/R -> 0 (signed)
    kappa = 0.0 if abs(R) < 1e-9 else (s / R)
    kappa_arr[i] = kappa

    # Lateral acceleration
    ay_i = v * v * kappa
    ay[i] = ay_i

    # Desired lean angle for comfort (full cancellation)
    phi_des_i = -math.atan(ay_i / g)
    phi_des_i = clip(phi_des_i, -phi_max, phi_max)
    phi_des[i] = phi_des_i

    # -----------------------------
    # Gain scheduling: schedule Kp and Krate based on (v, |kappa|)
    # -----------------------------
    kappa_mag = abs(kappa)
    Kp_sched = bilinear_2d(v, kappa_mag, v_grid, kappa_grid, Kp_table)
    Krate_sched = bilinear_2d(v, kappa_mag, v_grid, kappa_grid, Krate_table)

    # Clamp for safety
    Kp_sched = clip(Kp_sched, Kp_min, Kp_max)
    Krate_sched = clip(Krate_sched, Krate_min, Krate_max)

    Kp_used[i] = Kp_sched
    Krate_used[i] = Krate_sched

    # -----------------------------
    # Comfort metric (reduced-model): felt lateral accel ~ g*(phi_des - phi)
    # -----------------------------
    a_lat_feel[i] = g * (phi_des_i - phi[i])

    # -----------------------------
    # PI outer-loop using scheduled Kp
    # -----------------------------
    e = phi_des_i - phi[i]
    e_arr[i] = e
    e_int_arr[i] = e_int

    phi_dot_cmd_unsat = Kp_sched * e + Ki * e_int
    phi_dot_cmd_i = clip(phi_dot_cmd_unsat, -phi_dot_cmd_max, phi_dot_cmd_max)
    phi_dot_cmd[i] = phi_dot_cmd_i

    # Anti-windup
    saturated = abs(phi_dot_cmd_unsat - phi_dot_cmd_i) > 1e-12
    if (not saturated) or ((phi_dot_cmd_i == phi_dot_cmd_max and e < 0.0) or (phi_dot_cmd_i == -phi_dot_cmd_max and e > 0.0)):
        e_int += e * dt

    # -----------------------------
    # Feedforward + feedback torque (feedback uses scheduled Krate)
    # -----------------------------
    tau_ff_i = kphi * phi_des_i
    tau_ff[i] = tau_ff_i

    tau_fb_i = Krate_sched * (phi_dot_cmd_i - phi_dot[i])
    tau_fb[i] = tau_fb_i

    tau_act_i = tau_ff_i + tau_fb_i
    tau_act[i] = tau_act_i

    # -----------------------------
    # Plant dynamics:
    # I*phi_ddot + c*phi_dot + k*phi = tau_act
    # -----------------------------
    phi_ddot_i = (tau_act_i - cphi * phi_dot[i] - kphi * phi[i]) / Iphi

    # Integrate (semi-implicit Euler)
    phi_dot[i + 1] = phi_dot[i] + phi_ddot_i * dt
    phi[i + 1] = phi[i] + phi_dot[i + 1] * dt


# Fill last sample for plotting
phi_des[-1] = phi_des[-2]
phi_dot_cmd[-1] = phi_dot_cmd[-2]
tau_act[-1] = tau_act[-2]
tau_ff[-1] = tau_ff[-2]
tau_fb[-1] = tau_fb[-2]
ay[-1] = ay[-2]
kappa_arr[-1] = kappa_arr[-2]
e_arr[-1] = e_arr[-2]
e_int_arr[-1] = e_int
a_lat_feel[-1] = a_lat_feel[-2]
Kp_used[-1] = Kp_used[-2]
Krate_used[-1] = Krate_used[-2]


# -----------------------------
# PLOTS
# -----------------------------
plt.figure()
plt.plot(t, np.degrees(phi), label="phi (deg)")
plt.plot(t, np.degrees(phi_des), label="phi_des (deg)")
plt.axhline(phi_max_deg, linestyle="--")
plt.axhline(-phi_max_deg, linestyle="--")
plt.xlabel("time (s)")
plt.ylabel("roll angle (deg)")
plt.title("Roll angle tracking (comfort-first turn-in/out)")
plt.legend()
plt.grid(True)

plt.figure()
plt.plot(t, np.degrees(phi_dot), label="phi_dot (deg/s)")
plt.plot(t, np.degrees(phi_dot_cmd), label="phi_dot_cmd (deg/s)")
plt.axhline(phi_dot_cmd_max_deg_s, linestyle="--")
plt.axhline(-phi_dot_cmd_max_deg_s, linestyle="--")
plt.xlabel("time (s)")
plt.ylabel("roll rate (deg/s)")
plt.title("Roll rate and commanded roll rate")
plt.legend()
plt.grid(True)

plt.figure()
plt.plot(t, tau_act, label="tau_act = tau_ff + tau_fb (N*m)")
plt.plot(t, tau_ff, label="tau_ff (N*m)")
plt.plot(t, tau_fb, label="tau_fb (N*m)")
plt.xlabel("time (s)")
plt.ylabel("torque (N*m)")
plt.title("Actuator torques (comfort-first, effort allowed)")
plt.legend()
plt.grid(True)

plt.figure()
plt.plot(t, ay, label="a_y (m/s^2)")
plt.axhline(0.0, linestyle="--")
plt.xlabel("time (s)")
plt.ylabel("lateral acceleration (m/s^2)")
plt.title("Lateral acceleration profile (curvature ramp-in/out)")
plt.legend()
plt.grid(True)

plt.figure()
plt.plot(t, kappa_arr, label="curvature kappa (1/m)")
plt.axhline(0.0, linestyle="--")
plt.xlabel("time (s)")
plt.ylabel("curvature (1/m)")
plt.title("Curvature profile (0 → 1/R → 0)")
plt.legend()
plt.grid(True)

plt.figure()
plt.plot(t, np.degrees(e_arr), label="error e = phi_des - phi (deg)")
plt.plot(t, e_int_arr, label="integrator e_int (rad*s)")
plt.axhline(0.0, linestyle="--")
plt.xlabel("time (s)")
plt.ylabel("error / integrator")
plt.title("PI error and integrator state")
plt.legend()
plt.grid(True)

plt.figure()
plt.plot(t, a_lat_feel, label="felt lateral accel (m/s^2)")
plt.axhline(0.0, linestyle="--")
plt.xlabel("time (s)")
plt.ylabel("m/s^2")
plt.title("Passenger felt lateral acceleration (comfort metric)")
plt.legend()
plt.grid(True)

plt.figure()
plt.plot(t, a_lat_feel / g, label="felt lateral accel (g)")
plt.axhline(0.0, linestyle="--")
plt.xlabel("time (s)")
plt.ylabel("g")
plt.title("Passenger felt lateral acceleration (in g)")
plt.legend()
plt.grid(True)

# Scheduled gains plots
plt.figure()
plt.plot(t, Kp_used, label="Kp (scheduled)")
plt.xlabel("time (s)")
plt.ylabel("Kp")
plt.title("Scheduled Kp over time")
plt.grid(True)
plt.legend()

plt.figure()
plt.plot(t, Krate_used, label="Krate (scheduled)")
plt.xlabel("time (s)")
plt.ylabel("Krate")
plt.title("Scheduled Krate over time")
plt.grid(True)
plt.legend()

plt.show()
